# 05 — Final results, figures, and provenance gate

This model-free notebook renders the six paper figures, writes the canonical final metrics and results summary, extracts post-refactor provenance, and compares every scientific field to the pre-refactor snapshot. A difference outside the frozen tolerance raises an error rather than silently publishing a regression.

**Outputs:** `results/metrics.json`, `results/RESULTS.md`, six figures in both final figure directories, and post-refactor provenance/comparison files.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
ARTIFACTS = ROOT / 'artifacts' / 'final'
RESULTS = ROOT / 'results'
PAPER = ROOT / 'paper'
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'figures').mkdir(parents=True, exist_ok=True)
(PAPER / 'figures').mkdir(parents=True, exist_ok=True)

from src.evaluation import compare_provenance, extract_provenance
from src.plotting import generate_final_figures

def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

final_metrics = load_json(ARTIFACTS / '04_evaluation.json')
result_figures = generate_final_figures(final_metrics, RESULTS / 'figures')
paper_figures = generate_final_figures(final_metrics, PAPER / 'figures')
final_metrics['figures'] = {
    key: {
        'results_path': f"results/figures/{record['filename']}",
        'paper_path': f"paper/figures/{record['filename']}",
        'bytes': record['bytes'],
        'dpi': record['dpi'],
    }
    for key, record in result_figures.items()
}
metrics_path = RESULTS / 'metrics.json'
write_json(metrics_path, final_metrics)

source_paths = [
    metrics_path,
    ROOT / 'src' / 'jlens_interface.py',
    ROOT / 'src' / 'causal_read.py',
    ROOT / 'src' / 'cheap_read.py',
    ROOT / 'src' / 'datasets.py',
    ROOT / 'src' / 'evaluation.py',
    ROOT / 'src' / 'plotting.py',
]
source_hashes = {str(path.relative_to(ROOT)): sha256(path) for path in source_paths}
source_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
branch = subprocess.run(['git', 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
post = extract_provenance(
    final_metrics,
    snapshot='post_refactor',
    source_commit=source_commit,
    branch=branch,
    source_hashes=source_hashes,
)
post_path = RESULTS / 'PROVENANCE_post_refactor.json'
write_json(post_path, post)
pre = load_json(RESULTS / 'PROVENANCE_pre_refactor.json')
comparison = compare_provenance(pre, post, raise_on_regression=False)
comparison['pre_path'] = 'results/PROVENANCE_pre_refactor.json'
comparison['post_path'] = 'results/PROVENANCE_post_refactor.json'
comparison_path = RESULTS / 'PROVENANCE_comparison.json'
write_json(comparison_path, comparison)
if comparison['status'] != 'PASS':
    raise RuntimeError(f"Refactor regression: {comparison['regressions']}")

c = post['causal_sanity']
b = post['binary_detection']
g = post['graded_use']
d = post['read_distributions']
old_c_ci = final_metrics['old_binary']['causal_sanity']['median_intervals']
hard_c_ci = final_metrics['hard_control']['causal_sanity']['median_intervals']
results_markdown = f'''# Final results

## Verdict

**Binary relevant-vs-idle detector: supported. Graded causal-use meter: not supported. Final stress-test label: ARTIFACT (partial).**

On the frozen Qwen2.5-7B-Instruct experiment, READ_IG separates causally used explicit concepts from causally idle explicit concepts, including answer-type-matched controls. It does not rank causal magnitude among the already-strong engine examples.

## Scope and dataset

- Model: `Qwen/Qwen2.5-7B-Instruct` at pinned revision `{post['model']['revision']}` in bf16.
- Measurement: layer L{post['model']['source_layer']}, explicit single concept token, WRITTEN threshold `{post['model']['written_threshold']:.6f}`.
- Candidates: {post['dataset']['candidate_pairs']}; calibration: {post['dataset']['calibration_pairs']}; held-out evaluation: {post['dataset']['evaluation_pairs']}.
- Verified held-out pairs: {post['dataset']['verified_pairs']} in {post['dataset']['evaluation_dependency_groups']} dependency groups; {post['dataset']['unverified_pairs']} remained UNVERIFIED.
- Inference: five whole-group folds and {post['model']['bootstrap_draws']:,} dependency-group bootstrap draws, seed {post['model']['seed']}.

The labels used for ROC AUC are the constructed relevant-engine versus idle-control task classes. They are validated by causal interchange; they are not labels created by thresholding C.

## Causal sanity

| class | N | median signed C (grouped 95% CI) | median |C| (grouped 95% CI) |
| --- | ---: | --- | --- |
| relevant engine | {c['n_pairs']} | {c['engine_C_median']:.6f} [{old_c_ci['engine_C']['ci95_low']:.6f}, {old_c_ci['engine_C']['ci95_high']:.6f}] | {c['engine_abs_C_median']:.6f} [{old_c_ci['engine_abs_C']['ci95_low']:.6f}, {old_c_ci['engine_abs_C']['ci95_high']:.6f}] |
| original idle dashboard | {c['n_pairs']} | {c['old_dashboard_C_median']:.6f} [{old_c_ci['dashboard_C']['ci95_low']:.6f}, {old_c_ci['dashboard_C']['ci95_high']:.6f}] | {c['old_dashboard_abs_C_median']:.6f} [{old_c_ci['dashboard_abs_C']['ci95_low']:.6f}, {old_c_ci['dashboard_abs_C']['ci95_high']:.6f}] |
| answer-type-matched idle dashboard | {post['dataset']['hard_dashboard_verified']} | {c['hard_dashboard_C_median']:.6f} [{hard_c_ci['hard_dashboard_C']['ci95_low']:.6f}, {hard_c_ci['hard_dashboard_C']['ci95_high']:.6f}] | {c['hard_dashboard_abs_C_median']:.6f} [{hard_c_ci['hard_dashboard_abs_C']['ci95_low']:.6f}, {hard_c_ci['hard_dashboard_abs_C']['ci95_high']:.6f}] |

No class produced a preregistered sharp directional-disagreement flag. Full-residual C is signed and unclipped.

![Causal sanity](figures/f1_causal_sanity.png)

## Binary detection

| comparison / estimator | held-out AUC | grouped 95% CI |
| --- | ---: | --- |
| READ_IG, engine vs original idle | {b['READ_IG_old_dashboard_auc']:.6f} | [{b['READ_IG_old_dashboard_ci95_low']:.6f}, {b['READ_IG_old_dashboard_ci95_high']:.6f}] |
| READ_IG, engine vs answer-matched idle | {b['READ_IG_hard_dashboard_auc']:.6f} | [{b['READ_IG_hard_dashboard_ci95_low']:.6f}, {b['READ_IG_hard_dashboard_ci95_high']:.6f}] |
| READ_local, engine vs original idle | {b['READ_local_old_dashboard_auc']:.6f} | [{b['READ_local_old_dashboard_ci95_low']:.6f}, {b['READ_local_old_dashboard_ci95_high']:.6f}] |
| static capacity control | {b['capacity_baseline_auc']:.6f} | [{b['capacity_baseline_ci95_low']:.6f}, {b['capacity_baseline_ci95_high']:.6f}] |

The answer-type-matched control shows that arithmetic output type is not the sole source of binary separation. No wall-clock benchmark was run, so READ is described precisely as gradient-only, donor-free, and intervention-output-free—not as an empirically timed speedup.

![Binary AUC and baseline](figures/f2_binary_auc_and_baseline.png)

## Graded-use stress test

Within the {c['n_pairs']} engines, READ_IG has Spearman rho `{g['READ_IG_engine_only_rho']:.6f}` with |C|, with grouped 95% CI `[{g['READ_IG_engine_only_ci95_low']:.6f}, {g['READ_IG_engine_only_ci95_high']:.6f}]`. The CI spans zero and the point estimate is negative. Engine |C| occupies a narrow strong range from `{g['engine_abs_C_min']:.6f}` to `{g['engine_abs_C_max']:.6f}`. No weak/strong AUC was invented after inspection (`{g['within_engine_auc_status']}`).

The pooled rho `{g['READ_IG_overall_spearman_rho']:.6f}` is descriptive of the engine/control class gap and is not evidence that READ_IG resolves graded causal strength inside engines.

![Engine-only graded check](figures/f3_engine_only_graded_check.png)

## Raw READ_IG distributions

| class | min | median | max | IQR |
| --- | ---: | ---: | ---: | ---: |
| engine | {d['engine_min']:.6f} | {d['engine_median']:.6f} | {d['engine_max']:.6f} | {d['engine_iqr']:.6f} |
| original idle | {d['old_dashboard_min']:.6f} | {d['old_dashboard_median']:.6f} | {d['old_dashboard_max']:.6f} | {d['old_dashboard_iqr']:.6f} |
| answer-matched idle | {d['hard_dashboard_min']:.6f} | {d['hard_dashboard_median']:.6f} | {d['hard_dashboard_max']:.6f} | {d['hard_dashboard_iqr']:.6f} |

The two idle ranges overlap across `{100*d['old_hard_range_overlap_fraction']:.1f}%` of their union and both remain disjoint from engines on this roster.

![Hard-control AUC](figures/f4_hard_dashboard_auc.png)

![READ distributions](figures/f5_read_ig_distributions.png)

![Directional agreement](figures/f6_directional_agreement.png)

## Firewall and provenance

`src/cheap_read.py` imports no causal, patching, or intervention module. Notebook 03 consumes only the sanitized manifest and direction cache and freezes both old and hard-control READ values before notebook 04 computes hard causal truth.

The post-refactor comparison checked {comparison['compared_leaf_fields']} scientific fields with absolute/relative tolerance 1e-3: **{comparison['status']}**, with {comparison['regression_count']} regressions. See `PROVENANCE_pre_refactor.json`, `PROVENANCE_post_refactor.json`, and `PROVENANCE_comparison.json`.
'''
(RESULTS / 'RESULTS.md').write_text(results_markdown)
print(json.dumps({
    'decision': post['decision'],
    'provenance_comparison': comparison,
    'metrics_sha256': sha256(metrics_path),
    'figures': sorted(record['filename'] for record in result_figures.values()),
}, indent=2))


{
  "decision": {
    "binary_detector": "SUPPORTED",
    "graded_meter": "NOT_SUPPORTED",
    "stress_test_label": "ARTIFACT (partial)"
  },
  "provenance_comparison": {
    "status": "PASS",
    "absolute_tolerance": 0.001,
    "relative_tolerance": 0.001,
    "compared_leaf_fields": 80,
    "regression_count": 0,
    "regressions": [],
    "excluded_context_fields": [
      "snapshot",
      "source_commit",
      "branch",
      "source_hashes",
      "comparison_policy"
    ],
    "pre_path": "results/PROVENANCE_pre_refactor.json",
    "post_path": "results/PROVENANCE_post_refactor.json"
  },
  "metrics_sha256": "8ff43f720635f466eabbf0374a88aab241a8ec58b5def049e568579ae3f9f691",
  "figures": [
    "f1_causal_sanity.png",
    "f2_binary_auc_and_baseline.png",
    "f3_engine_only_graded_check.png",
    "f4_hard_dashboard_auc.png",
    "f5_read_ig_distributions.png",
    "f6_directional_agreement.png"
  ]
}
